In [1]:
%pip install seaborn lightgbm

Note: you may need to restart the kernel to use updated packages.


# Titanic ML Pipeline - modeling_ddb

DynamoDB에서 데이터를 로드하고, 학습 결과를 모두 DynamoDB에 저장합니다.

`modeling.ipynb`의 DynamoDB 버전. S3/로컬 파일 쓰기 없음.

## 1. 환경 설정 및 Import

In [20]:
import io
import sys
import os
import yaml
import json
import hashlib
import pickle
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # 파일 저장 없이 메모리 렌더링
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime
from uuid import uuid4

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, log_loss, confusion_matrix, roc_curve,
    classification_report
)
import lightgbm as lgb

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')

BASE_DIR = Path.cwd()
sys.path.insert(0, str(BASE_DIR))
from ddb_store import DDBStore

print('라이브러리 임포트 완료')

라이브러리 임포트 완료


## 2. 설정 로드 (DynamoDB)

In [21]:
# 실험 식별 정보
REGION     = 'us-east-1'
USER_ID    = 'hjsong'
PROJECT    = 'titanic-survival-prediction'
EXPERIMENT = 'tuned-hjsong-v1'

store  = DDBStore(region=REGION)
EXP_PK = DDBStore.make_exp_pk(USER_ID, PROJECT, EXPERIMENT)

# DynamoDB에서 config 로드
conf_item    = store.get_experiment_conf(EXP_PK)
env_config   = conf_item['env_yml']
meta_config  = conf_item['meta_yml']
model_config = conf_item['model_yml']

VERSION    = meta_config['version']
ENV        = env_config['env']
PROJECT    = meta_config['project']
EXPERIMENT = meta_config['experiment']

print(f'EXP_PK: {EXP_PK}')
print(f'Project: {meta_config["project"]}')
print(f'Experiment: {meta_config["experiment"]}')
print(f'Algorithm: {model_config["algorithm"]["name"]}')

EXP_PK: EXP#hjsong#titanic-survival-prediction#tuned-hjsong-v1
Project: titanic-survival-prediction
Experiment: tuned-hjsong-v1
Algorithm: lightgbm


In [22]:
# 이 셀에 반드시 'parameters' 태그 달아야 함
run_id     = None   # run_pm.py에서 주입됨
output_dir = None   # (DDB 버전에서는 사용하지 않음)

In [23]:
# Run ID 생성
RUN_DATE  = datetime.now().strftime('%Y%m%d')
RUN_UUID  = uuid4().hex[:8]
ALGORITHM = model_config['algorithm']['name']
SUFFIX    = model_config['algorithm']['suffix']
RUN_ID    = run_id if run_id is not None else f'{RUN_DATE}_{ALGORITHM}_{SUFFIX}_{RUN_UUID}'
CREATED_AT = datetime.now().isoformat()

RUN_PK = DDBStore.make_run_pk(USER_ID, PROJECT, EXPERIMENT, RUN_ID)

print(f'RUN_ID : {RUN_ID}')
print(f'RUN_PK : {RUN_PK}')

RUN_ID : 20260326_lightgbm_baseline_75023c7e
RUN_PK : RUN#hjsong#titanic-survival-prediction#tuned-hjsong-v1#20260326_lightgbm_baseline_75023c7e


## 3. 데이터 로드 (DynamoDB)

In [24]:
# DynamoDB에서 DataFrame으로 직접 로드
train_df = store.get_dataset_split(EXP_PK, 'train')
test_df  = store.get_dataset_split(EXP_PK, 'test')

try:
    valid_df = store.get_dataset_split(EXP_PK, 'validation')
    print(f'validation: {len(valid_df)}rows')
except KeyError:
    valid_df = None
    print('validation 없음 - train에서 분할 예정')

print(f'train: {len(train_df)}rows')
print(f'test : {len(test_df)}rows')

validation: 418rows
train: 891rows
test : 418rows


## 4. 전처리

In [25]:
def preprocess_data(df, is_train=True):
    df = df.copy()
    features_config = model_config['features']
    target_col      = features_config['target_col']
    numeric_cols    = features_config['numeric_col']

    # 결측치
    df['Age']      = df['Age'].fillna(df['Age'].median())
    df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])
    df['Fare']     = df['Fare'].fillna(df['Fare'].median())

    # 파생 피처
    df['FamilySize']   = df['SibSp'] + df['Parch'] + 1
    df['IsAlone']      = (df['FamilySize'] == 1).astype(int)
    df['FarePerPerson'] = df['Fare'] / df['FamilySize']

    # 인코딩
    df['Sex'] = df['Sex'].map({'male': 1, 'female': 0})
    embarked_dummies = pd.get_dummies(df['Embarked'], prefix='Embarked')
    df = pd.concat([df, embarked_dummies], axis=1)

    feature_cols  = numeric_cols + ['Sex', 'Pclass', 'FamilySize', 'IsAlone', 'FarePerPerson']
    feature_cols += [c for c in df.columns if c.startswith('Embarked_')]

    X = df[feature_cols]
    y = df[target_col] if target_col in df.columns else None
    return X, y, feature_cols

print('전처리 함수 정의 완료')

전처리 함수 정의 완료


In [26]:
X_train_full, y_train_full, feature_cols = preprocess_data(train_df, is_train=True)

SEED       = model_config['data']['seed']
VALID_RATIO = model_config['data']['train_valid_split']

X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_full, y_train_full,
    test_size=VALID_RATIO, random_state=SEED, stratify=y_train_full
)

print(f'Train: {len(X_train)}samples')
print(f'Valid: {len(X_valid)}samples')
print(f'Features: {feature_cols}')

Train: 712samples
Valid: 179samples
Features: ['Age', 'Fare', 'SibSp', 'Parch', 'Sex', 'Pclass', 'FamilySize', 'IsAlone', 'FarePerPerson', 'Embarked_C', 'Embarked_Q', 'Embarked_S']


## 5. 모델 학습

In [27]:
train_data = lgb.Dataset(X_train, label=y_train)
valid_data = lgb.Dataset(X_valid, label=y_valid, reference=train_data)

params = {
    'objective':        model_config['hyperparameters']['objective'],
    'boosting':         model_config['hyperparameters']['boosting'],
    'max_depth':        model_config['hyperparameters']['max_depth'],
    'learning_rate':    model_config['hyperparameters']['learning_rate'],
    'num_leaves':       model_config['hyperparameters']['num_leaves'],
    'feature_fraction': model_config['hyperparameters']['feature_fraction'],
    'bagging_fraction': model_config['hyperparameters']['bagging_fraction'],
    'bagging_freq':     model_config['hyperparameters']['bagging_freq'],
    'verbosity': -1,
    'seed': SEED,
    'metric': ['auc', 'binary_logloss'],
}
print('하이퍼파라미터 설정 완료')

하이퍼파라미터 설정 완료


In [28]:
evals_result = {}
print('모델 학습 시작...')
model = lgb.train(
    params, train_data,
    num_boost_round=model_config['hyperparameters']['num_iterations'],
    valid_sets=[train_data, valid_data],
    valid_names=['train', 'valid'],
    callbacks=[lgb.log_evaluation(period=50), lgb.record_evaluation(evals_result)]
)
COMPLETED_AT = datetime.now().isoformat()
print(f'학습 완료. Best iteration: {model.best_iteration}')

모델 학습 시작...
[50]	train's auc: 0.930578	train's binary_logloss: 0.343151	valid's auc: 0.85639	valid's binary_logloss: 0.427387
[100]	train's auc: 0.951592	train's binary_logloss: 0.282467	valid's auc: 0.866864	valid's binary_logloss: 0.426673
[150]	train's auc: 0.966507	train's binary_logloss: 0.245307	valid's auc: 0.860804	valid's binary_logloss: 0.44404
[200]	train's auc: 0.974668	train's binary_logloss: 0.217155	valid's auc: 0.865152	valid's binary_logloss: 0.452804
학습 완료. Best iteration: 0


## 6. 모델 평가

In [29]:
def evaluate_model(model, X, y, name=''):
    y_proba = model.predict(X)
    y_pred  = (y_proba >= 0.5).astype(int)
    metrics = {
        'accuracy':  round(accuracy_score(y, y_pred), 4),
        'precision': round(precision_score(y, y_pred), 4),
        'recall':    round(recall_score(y, y_pred), 4),
        'f1_score':  round(f1_score(y, y_pred), 4),
        'auc_roc':   round(roc_auc_score(y, y_proba), 4),
        'log_loss':  round(log_loss(y, y_proba), 4),
        'samples':   int(len(y)),
    }
    print(f'{name}: accuracy={metrics["accuracy"]}, auc={metrics["auc_roc"]}')
    return metrics, y_pred, y_proba

train_metrics, y_train_pred, y_train_proba = evaluate_model(model, X_train, y_train, 'Train')
valid_metrics, y_valid_pred, y_valid_proba = evaluate_model(model, X_valid, y_valid, 'Valid')

Train: accuracy=0.9199, auc=0.9747
Valid: accuracy=0.8324, auc=0.8652


In [30]:
cm = confusion_matrix(y_valid, y_valid_pred)
tn, fp, fn, tp = cm.ravel()
print(f'Confusion Matrix - TN:{tn} FP:{fp} FN:{fn} TP:{tp}')

Confusion Matrix - TN:100 FP:10 FN:20 TP:49


In [31]:
feature_importance = pd.DataFrame({
    'feature':    feature_cols,
    'importance': model.feature_importance(importance_type='gain')
}).sort_values('importance', ascending=False)
feature_importance['importance'] = feature_importance['importance'] / feature_importance['importance'].sum()
feature_importance['rank'] = range(1, len(feature_importance) + 1)
print(feature_importance.head(10).to_string(index=False))

      feature  importance  rank
          Sex    0.299821     1
          Age    0.187071     2
         Fare    0.181146     3
FarePerPerson    0.166054     4
       Pclass    0.080783     5
   FamilySize    0.037067     6
   Embarked_S    0.016134     7
        SibSp    0.011351     8
        Parch    0.008275     9
   Embarked_Q    0.005316    10


---
## 7. 결과 저장 (DynamoDB)

### 7.1 Config 스냅샷 저장

In [32]:
store.put_run_config_snapshot(
    run_pk=RUN_PK, exp_pk=EXP_PK,
    env_config=env_config, meta_config=meta_config, model_config=model_config,
)
print(f'[OK] CONFIG: {RUN_PK}')

[OK] CONFIG: RUN#hjsong#titanic-survival-prediction#tuned-hjsong-v1#20260326_lightgbm_baseline_75023c7e


### 7.2 차트 생성 (메모리) + 저장

In [33]:
charts_bytes = {}

# Feature Importance
fig, ax = plt.subplots(figsize=(10, 8))
sns.barplot(data=feature_importance.head(10), x='importance', y='feature', palette='Blues_d', ax=ax)
ax.set_title('Feature Importance (Top 10)', fontsize=14, fontweight='bold')
plt.tight_layout()
buf = io.BytesIO(); fig.savefig(buf, format='png', dpi=150, bbox_inches='tight'); plt.close()
charts_bytes['feature_importance'] = buf.getvalue()
print('  [OK] feature_importance')

# ROC Curve
fpr, tpr, _ = roc_curve(y_valid, y_valid_proba)
auc_score   = roc_auc_score(y_valid, y_valid_proba)
fig, ax = plt.subplots(figsize=(8, 8))
ax.plot(fpr, tpr, 'b-', linewidth=2, label=f'AUC={auc_score:.4f}')
ax.plot([0,1],[0,1],'k--')
ax.fill_between(fpr, tpr, alpha=0.3)
ax.set_title('ROC Curve', fontsize=14, fontweight='bold'); ax.legend()
plt.tight_layout()
buf = io.BytesIO(); fig.savefig(buf, format='png', dpi=150, bbox_inches='tight'); plt.close()
charts_bytes['roc_curve'] = buf.getvalue()
print('  [OK] roc_curve')

# Confusion Matrix
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Died','Survived'], yticklabels=['Died','Survived'])
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
buf = io.BytesIO(); fig.savefig(buf, format='png', dpi=150, bbox_inches='tight'); plt.close()
charts_bytes['confusion_matrix'] = buf.getvalue()
print('  [OK] confusion_matrix')

# Learning Curve
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(evals_result['train']['auc'], label='Train')
axes[0].plot(evals_result['valid']['auc'], label='Valid')
axes[0].set_title('AUC'); axes[0].legend()
axes[1].plot(evals_result['train']['binary_logloss'], label='Train')
axes[1].plot(evals_result['valid']['binary_logloss'], label='Valid')
axes[1].set_title('Log Loss'); axes[1].legend()
plt.tight_layout()
buf = io.BytesIO(); fig.savefig(buf, format='png', dpi=150, bbox_inches='tight'); plt.close()
charts_bytes['learning_curve'] = buf.getvalue()
print('  [OK] learning_curve')

# Feature Impact (Explainability)
top_10 = feature_importance.head(10)
colors = ['#ff7f0e' if i%2==0 else '#1f77b4' for i in range(len(top_10))]
fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(range(len(top_10)), top_10['importance'].values, color=colors)
ax.set_yticks(range(len(top_10))); ax.set_yticklabels(top_10['feature'].values)
ax.invert_yaxis(); ax.set_title('Feature Impact Summary', fontsize=14, fontweight='bold')
plt.tight_layout()
buf = io.BytesIO(); fig.savefig(buf, format='png', dpi=150, bbox_inches='tight'); plt.close()
charts_bytes['feature_impact_summary'] = buf.getvalue()
print('  [OK] feature_impact_summary')

# DDB 저장
store.put_charts(run_pk=RUN_PK, exp_pk=EXP_PK, charts_bytes=charts_bytes)
print(f'[OK] CHARTS + EXPLAINABILITY: {RUN_PK}')

  [OK] feature_importance
  [OK] roc_curve
  [OK] confusion_matrix
  [OK] learning_curve
  [OK] feature_impact_summary
[OK] CHARTS + EXPLAINABILITY: RUN#hjsong#titanic-survival-prediction#tuned-hjsong-v1#20260326_lightgbm_baseline_75023c7e


### 7.3 모델 저장 (청킹)

In [34]:
import math
pkl_size_mb = len(pickle.dumps(model)) / (1024*1024)
print(f'model.pkl 크기: {pkl_size_mb:.2f}MB')

total_chunks = store.put_model_chunked(
    run_pk=RUN_PK, exp_pk=EXP_PK,
    model_obj=model, algorithm=ALGORITHM, suffix=SUFFIX,
)
print(f'[OK] MODEL: {total_chunks}개 청크로 저장 -> {RUN_PK}')

model.pkl 크기: 0.34MB
[OK] MODEL: 2개 청크로 저장 -> RUN#hjsong#titanic-survival-prediction#tuned-hjsong-v1#20260326_lightgbm_baseline_75023c7e


### 7.4 지표 저장 (METRICS)

In [35]:
model_metrics = {
    'run_id':      RUN_ID,
    'model_name':  meta_config['model'],
    'algorithm':   ALGORITHM,
    'task':        'binary_classification',
    'evaluated_at': COMPLETED_AT,
    'train_metrics': train_metrics,
    'valid_metrics': valid_metrics,
    'confusion_matrix': {
        'true_negative':  int(tn), 'false_positive': int(fp),
        'false_negative': int(fn), 'true_positive':  int(tp),
        'specificity': round(float(tn / (tn + fp)), 4),
        'sensitivity': round(float(tp / (tp + fn)), 4),
    },
    'feature_importance': [
        {'feature': row['feature'], 'importance': round(float(row['importance']), 4), 'rank': int(row['rank'])}
        for _, row in feature_importance.head(10).iterrows()
    ],
    'training_history': {
        'best_iteration':   model.best_iteration,
        'final_train_auc':  round(evals_result['train']['auc'][-1], 4),
        'final_valid_auc':  round(evals_result['valid']['auc'][-1], 4),
    },
    'model_complexity': {
        'num_trees':    model.num_trees(),
        'max_depth':    params['max_depth'],
        'num_leaves':   params['num_leaves'],
        'total_features': len(feature_cols),
        'pkl_size_mb':  round(pkl_size_mb, 2),
    },
}
store.put_run_metrics(run_pk=RUN_PK, exp_pk=EXP_PK, metrics=model_metrics)
print(f'[OK] METRICS: {RUN_PK}')

[OK] METRICS: RUN#hjsong#titanic-survival-prediction#tuned-hjsong-v1#20260326_lightgbm_baseline_75023c7e


### 7.5 데이터 참조 저장 (DATA_REF)

In [36]:
input_data_ref = {
    'source': {
        'type':    'dynamodb',
        'table':   'gsretail-mlops-edu-hjsong',
        'exp_pk':  EXP_PK,
        'version': VERSION,
    },
    'split_keys': {
        'train':      f'{EXP_PK}|DATA#train',
        'validation': f'{EXP_PK}|DATA#validation',
        'test':       f'{EXP_PK}|DATA#test',
    },
    'schema': {
        'index_col':     model_config['features']['index_col'],
        'target_col':    model_config['features']['target_col'],
        'feature_count': len(feature_cols),
        'feature_cols':  feature_cols,
    },
    'split': {
        'method':       'stratified_random',
        'train_ratio':  1 - VALID_RATIO,
        'valid_ratio':  VALID_RATIO,
        'seed':         SEED,
        'train_samples': int(len(X_train)),
        'valid_samples': int(len(X_valid)),
        'train_target_dist': {str(k): int(v) for k,v in y_train.value_counts().sort_index().items()},
        'valid_target_dist': {str(k): int(v) for k,v in y_valid.value_counts().sort_index().items()},
    },
}
store.put_run_data_ref(run_pk=RUN_PK, exp_pk=EXP_PK, ref=input_data_ref)
print(f'[OK] DATA_REF: {RUN_PK}')

[OK] DATA_REF: RUN#hjsong#titanic-survival-prediction#tuned-hjsong-v1#20260326_lightgbm_baseline_75023c7e


### 7.6 실행 매니페스트 저장 (MANIFEST)

In [37]:
import sys as _sys
duration = int((datetime.fromisoformat(COMPLETED_AT) - datetime.fromisoformat(CREATED_AT)).total_seconds())

run_manifest = {
    'run_id':       RUN_ID,
    'run_name':     f'{EXPERIMENT}-{ALGORITHM}-{SUFFIX}',
    'created_at':   CREATED_AT,
    'completed_at': COMPLETED_AT,
    'status':       'completed',
    'context': {
        'env':        ENV,
        'user_id':    USER_ID,
        'project':    PROJECT,
        'experiment': EXPERIMENT,
        'model':      meta_config['model'],
        'algorithm':  ALGORITHM,
        'version':    VERSION,
    },
    'storage': {'type': 'dynamodb', 'table': 'gsretail-mlops-edu-hjsong'},
    'runtime': {
        'platform':       'local',
        'python_version': f'{_sys.version_info.major}.{_sys.version_info.minor}',
        'framework':      f'lightgbm=={lgb.__version__}',
    },
    'summary': {
        'duration_seconds': duration,
        'train_samples':    int(len(X_train)),
        'valid_samples':    int(len(X_valid)),
        'feature_count':    len(feature_cols),
        'best_iteration':   model.best_iteration,
    },
    'ddb_items': ['MANIFEST','METRICS','DATA_REF','CONFIG',
                  f'MODEL (x{total_chunks} chunks)','CHARTS','EXPLAINABILITY','REPORT'],
}
store.put_run_manifest(run_pk=RUN_PK, exp_pk=EXP_PK, manifest=run_manifest)
print(f'[OK] MANIFEST: {RUN_PK}')

[OK] MANIFEST: RUN#hjsong#titanic-survival-prediction#tuned-hjsong-v1#20260326_lightgbm_baseline_75023c7e


### 7.7 HTML Report 저장 (REPORT)

In [38]:
html_report = f"""
<!DOCTYPE html><html lang="ko"><head>
<meta charset="UTF-8">
<title>Classification Report - {RUN_ID}</title>
<style>
body{{font-family:Arial,sans-serif;margin:40px;background:#f5f5f5;}}
.container{{max-width:1200px;margin:0 auto;background:white;padding:40px;border-radius:8px;}}
h1{{color:#2E75B6;border-bottom:3px solid #2E75B6;padding-bottom:10px;}}
.grid{{display:grid;grid-template-columns:repeat(4,1fr);gap:20px;margin:20px 0;}}
.card{{background:#f8f9fa;padding:20px;border-radius:8px;text-align:center;}}
.val{{font-size:2em;font-weight:bold;color:#2E75B6;}}
table{{width:100%;border-collapse:collapse;margin:20px 0;}}
th,td{{padding:12px;text-align:left;border-bottom:1px solid #ddd;}}
th{{background:#2E75B6;color:white;}}
</style></head><body><div class="container">
<h1>Titanic Classification Report</h1>
<p><b>Run ID:</b> {RUN_ID} | <b>Experiment:</b> {EXPERIMENT} | <b>Algorithm:</b> {ALGORITHM}</p>
<h2>Validation Metrics</h2>
<div class="grid">
  <div class="card"><div class="val">{valid_metrics['accuracy']:.1%}</div><div>Accuracy</div></div>
  <div class="card"><div class="val">{valid_metrics['precision']:.1%}</div><div>Precision</div></div>
  <div class="card"><div class="val">{valid_metrics['recall']:.1%}</div><div>Recall</div></div>
  <div class="card"><div class="val">{valid_metrics['auc_roc']:.4f}</div><div>AUC-ROC</div></div>
</div>
<h2>Feature Importance (Top 10)</h2>
<table><tr><th>Rank</th><th>Feature</th><th>Importance</th></tr>
{''.join(f"<tr><td>{int(r['rank'])}</td><td>{r['feature']}</td><td>{r['importance']:.4f}</td></tr>" for _,r in feature_importance.head(10).iterrows())}
</table>
</div></body></html>
"""
store.put_report(run_pk=RUN_PK, exp_pk=EXP_PK, html_content=html_report)
print(f'[OK] REPORT: {RUN_PK}')

[OK] REPORT: RUN#hjsong#titanic-survival-prediction#tuned-hjsong-v1#20260326_lightgbm_baseline_75023c7e


---
## 8. 검증

In [39]:
import boto3
from boto3.dynamodb.conditions import Key

ddb   = boto3.resource('dynamodb', region_name=REGION)
table = ddb.Table('gsretail-mlops-edu-hjsong')

print(f'RUN PK: {RUN_PK}')
print('=' * 70)
resp  = table.query(KeyConditionExpression=Key('experiment_id').eq(RUN_PK))
for item in sorted(resp['Items'], key=lambda x: x['entity_type']):
    size_kb = len(str(item)) / 1024
    print(f'  {item["entity_type"]:<30} ~{size_kb:.1f}KB')
print('=' * 70)
print(f'총 {len(resp["Items"])}개 아이템 저장 완료')

RUN PK: RUN#hjsong#titanic-survival-prediction#tuned-hjsong-v1#20260326_lightgbm_baseline_75023c7e
  CHARTS                         ~300.2KB
  CONFIG                         ~3.6KB
  DATA_REF                       ~1.2KB
  EXPLAINABILITY                 ~54.1KB
  MANIFEST                       ~1.1KB
  METRICS                        ~2.2KB
  MODEL#chunk_000                ~244.5KB
  MODEL#chunk_001                ~222.8KB
  REPORT                         ~2.2KB
총 9개 아이템 저장 완료


## 9. 최종 요약

In [40]:
print('=' * 60)
print('DynamoDB 모델링 완료')
print('=' * 60)
print(f'Run ID  : {RUN_ID}')
print(f'EXP_PK  : {EXP_PK}')
print(f'RUN_PK  : {RUN_PK}')
print()
print(f'Valid Accuracy : {valid_metrics["accuracy"]:.4f}')
print(f'Valid AUC-ROC  : {valid_metrics["auc_roc"]:.4f}')
print()
print('모델 로드 방법:')
print(f'  model = store.get_model("{RUN_PK}")')
print()
print(f'완료: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')

DynamoDB 모델링 완료
Run ID  : 20260326_lightgbm_baseline_75023c7e
EXP_PK  : EXP#hjsong#titanic-survival-prediction#tuned-hjsong-v1
RUN_PK  : RUN#hjsong#titanic-survival-prediction#tuned-hjsong-v1#20260326_lightgbm_baseline_75023c7e

Valid Accuracy : 0.8324
Valid AUC-ROC  : 0.8652

모델 로드 방법:
  model = store.get_model("RUN#hjsong#titanic-survival-prediction#tuned-hjsong-v1#20260326_lightgbm_baseline_75023c7e")

완료: 2026-03-26 09:48:34
